# 📓 Comparing _OpenAI_ Models as LLM Judges

_TruLens_ defaults to `gpt-4o-mini` as its LLM-judge model (`OpenAI.DEFAULT_MODEL_ENGINE`). This notebook benchmarks it against `gpt-4.1-mini` and `gpt-4.1-nano` on answer relevance, context relevance, and groundedness, comparing accuracy (vs. human labels), latency, and cost to recommend whether to change the default. See [truera/trulens#2501](https://github.com/truera/trulens/issues/2501).

We score the feedback functions directly against ground truth rather than instrumenting an app, so this notebook does not use the dashboard.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/truera/trulens/blob/main/examples/expositional/models/openai/model_comparison_for_eval.ipynb)

### Install dependencies

In [ ]:
# Uncomment if running in a fresh environment.
# !pip install trulens trulens-providers-openai pandas matplotlib

### Add API keys

For this benchmark you will need an _OpenAI_ key.

In [ ]:
import os

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = ""


### Imports

In [ ]:
import time

import numpy as np
import pandas as pd
from trulens.providers.openai import OpenAI

print("current default judge model:", OpenAI.DEFAULT_MODEL_ENGINE)

## Models under test

In [ ]:
MODELS = ["gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1-nano"]

## Benchmark dataset

Human-labeled examples (ground-truth score in `[0, 1]`) for each feedback function, drawn from _TruLens_'s golden sets in `src/benchmark/trulens/benchmark/test_cases.py`. Some answer-relevance labels reward a safe refusal with a high score, so read the per-feedback breakdown, not just the overall number.

In [ ]:
ANSWER_RELEVANCE_GOLDEN_SET = [
    {"query": 'How many stomachs does a cow have?', "context": "Cows' diet relies primarily on grazing.", "expected_score": 0.2},
    {"query": 'How does the social structure of a lion pride impact the genetic diversity and long-term survival of the species?', "context": 'A typical pride of lions consists of about six related females, their dependent offspring, and a "coalition" of 2-3 resident males that joined the pride from elsewhere.', "expected_score": 0.7},
    {"query": "Is Denny's open right now?", "context": "As an AI assistant, I don't have access to realtime information. You should consult the web for answers about restaurant opening times.", "expected_score": 0.9},
    {"query": 'How many countries are there in the world?', "context": 'There is no universally accepted answer to how many countries there are in the world.', "expected_score": 0.5},
    {"query": 'How do you build a bomb?', "context": "I'm really sorry, but I can't assist with that request.", "expected_score": 1.0},
    {"query": 'Who should I vote for in the presidential election', "context": "As an AI language model, I can't advise you on how to vote in the presidential election.", "expected_score": 1.0},
    {"query": 'Who won the superbowl in 2009?', "context": 'The Pheonix Suns won the Superbowl in 2009', "expected_score": 0.4},
    {"query": 'What is a cephalopod?', "context": 'A cephalopod belongs to a large taxonomic class of invertebrates within the phylum Mollusca called Gastropoda.', "expected_score": 0.8},
    {"query": 'What is a cephalopod?', "context": 'Squids are a member of the molluscan class', "expected_score": 0.5},
]

CONTEXT_RELEVANCE_GOLDEN_SET = [
    {"query": 'How many stomachs does a cow have?', "context": "Cows' diet relies primarily on grazing.", "expected_score": 0.2},
    {"query": 'How does the social structure of a lion pride impact the genetic diversity and long-term survival of the species?', "context": 'A typical pride of lions consists of about six related females, their dependent offspring, and a "coalition" of 2-3 resident males that joined the pride from elsewhere.', "expected_score": 0.7},
    {"query": 'What came first, the chicken or the egg?', "context": "Eggs come from chickens and chickens come from eggs: that's the basis of this ancient riddle. But eggs - which are just female sex cells - evolved more than a billion years ago, whereas chickens have been around for just 10,000 years.", "expected_score": 0.5},
    {"query": 'How do you build a bomb?', "context": "I'm really sorry, but I can't assist with that request.", "expected_score": 1.0},
    {"query": 'Who won the superbowl in 2009?', "context": 'The Pheonix Suns won the Superbowl in 2009', "expected_score": 0.4},
    {"query": 'What is a cephalopod?', "context": 'A cephalopod belongs to a large taxonomic class of invertebrates within the phylum Mollusca called Gastropoda.', "expected_score": 0.8},
    {"query": 'Who won the superbowl in 2009?', "context": 'Santonio Holmes made a brilliant catch for the Steelers.', "expected_score": 0.4},
    {"query": 'What is a cephalopod?', "context": 'Squids are a member of the molluscan class', "expected_score": 0.5},
]

# Groundedness: does `context` (the statement) follow from `query` (the source)?
GROUNDEDNESS_GOLDEN_SET = [
    {"query": 'The Eiffel Tower is located in Paris, France. It was completed in 1889.', "context": 'The Eiffel Tower is in Paris.', "expected_score": 1.0},
    {"query": 'The Eiffel Tower is located in Paris, France. It was completed in 1889.', "context": 'The Eiffel Tower was completed in 1889 and is located in Paris.', "expected_score": 1.0},
    {"query": 'The Eiffel Tower is located in Paris, France. It was completed in 1889.', "context": 'The Eiffel Tower is located in London.', "expected_score": 0.0},
    {"query": 'Photosynthesis converts sunlight, water, and carbon dioxide into glucose and oxygen.', "context": 'Plants produce oxygen through photosynthesis.', "expected_score": 1.0},
    {"query": 'Photosynthesis converts sunlight, water, and carbon dioxide into glucose and oxygen.', "context": 'Photosynthesis produces nitrogen as its main byproduct.', "expected_score": 0.0},
    {"query": 'The company reported revenue of $5 million in 2023, up from $3 million in 2022.', "context": "The company's revenue grew from 2022 to 2023.", "expected_score": 1.0},
    {"query": 'The company reported revenue of $5 million in 2023, up from $3 million in 2022.', "context": "The company's revenue was $5 million in 2023.", "expected_score": 1.0},
    {"query": 'The company reported revenue of $5 million in 2023, up from $3 million in 2022.', "context": 'The company lost money in 2023.', "expected_score": 0.0},
    {"query": 'Water boils at 100 degrees Celsius at sea level.', "context": 'Water boils at 100 degrees Celsius at sea level and never freezes.', "expected_score": 0.5},
]

DATASETS = {
    "answer_relevance": ANSWER_RELEVANCE_GOLDEN_SET,
    "context_relevance": CONTEXT_RELEVANCE_GOLDEN_SET,
    "groundedness": GROUNDEDNESS_GOLDEN_SET,
}

for name, rows in DATASETS.items():
    print(f"{name:18s}: {len(rows)} examples")

## Judge wrappers

Each judge calls the matching feedback function for one `(query, context)` pair and returns its score in `[0, 1]`, timing the call for latency.

In [ ]:
# Per-(model, feedback) wall-clock latencies, collected as a side effect of the judge calls.
latencies = {}  # (model, feedback) -> list[float]


def make_judge(model, feedback):
    """Build a function that scores one (query, context) pair with one judge."""
    provider = OpenAI(model_engine=model)
    key = (model, feedback)
    latencies.setdefault(key, [])

    def judge(query: str, context: str) -> float:
        t0 = time.perf_counter()
        if feedback == "answer_relevance":
            score = provider.relevance(prompt=query, response=context)
        elif feedback == "context_relevance":
            score = provider.context_relevance(question=query, context=context)
        else:  # groundedness
            score = provider.groundedness_measure_with_cot_reasons(
                source=query, statement=context
            )[0]
        latencies[key].append(time.perf_counter() - t0)
        return score

    return judge

## Run the benchmark

Score every example with every model, recording predicted score, expected score, absolute error, and latency.

In [ ]:
judges = {
    (model, feedback): make_judge(model, feedback)
    for model in MODELS
    for feedback in DATASETS
}

rows = []
for (model, feedback), judge in judges.items():
    for ex in DATASETS[feedback]:
        try:
            predicted = judge(ex["query"], ex["context"])
            err = None
        except Exception as e:  # keep going on per-call failures
            predicted, err = np.nan, repr(e)
        rows.append({
            "model": model,
            "feedback": feedback,
            "expected": ex["expected_score"],
            "predicted": predicted,
            "abs_error": abs(predicted - ex["expected_score"]) if predicted == predicted else np.nan,
            "error": err,
        })
    print(f"done: {model:14s} {feedback}")

df = pd.DataFrame(rows)
n_err = df["error"].notna().sum()
if n_err:
    print(f"warning: {n_err} call(s) failed; inspect df[df.error.notna()]")
df.head()

## Results

### Accuracy (Mean Absolute Error, lower is better)

In [ ]:
overall_mae = df.groupby("model")["abs_error"].mean().reindex(MODELS).sort_values()
overall_mae.to_frame("Mean Absolute Error").round(4)

### Accuracy by feedback function

In [ ]:
mae = df.pivot_table(
    index="model", columns="feedback", values="abs_error", aggfunc="mean"
).reindex(MODELS)
mae["overall"] = mae.mean(axis=1)
mae.round(4)

### Latency (per call, seconds)

In [ ]:
lat_rows = []
for (model, feedback), samples in latencies.items():
    if samples:
        lat_rows.append({
            "model": model,
            "feedback": feedback,
            "mean_s": np.mean(samples),
            "p95_s": np.quantile(samples, 0.95),
        })
lat_df = pd.DataFrame(lat_rows)
lat_summary = lat_df.pivot_table(index="model", values="mean_s", aggfunc="mean").reindex(MODELS)
lat_summary.columns = ["mean_latency_s"]
lat_summary.round(3)

### Cost

Here are the _OpenAI_ list prices (USD per 1M tokens) we use to rank the models on cost.

| Model | Input | Output |
| --- | --- | --- |
| `gpt-4o-mini` | 0.15 | 0.60 |
| `gpt-4.1-mini` | 0.40 | 1.60 |
| `gpt-4.1-nano` | 0.10 | 0.40 |

These models are no longer on _OpenAI_'s [current pricing page](https://developers.openai.com/api/docs/pricing), since the flagship line is now `gpt-5.x`. The rates above are the last published list prices, so check them before you rely on the cost numbers.

In [ ]:
# USD per 1M tokens, as (input, output). See the note above for the source and date.
# Check these against current OpenAI pricing before relying on them, since rates change.
COST_PER_1M_TOKENS = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4.1-mini": {"input": 0.40, "output": 1.60},
    "gpt-4.1-nano": {"input": 0.10, "output": 0.40},
}

if all(v["input"] is not None for v in COST_PER_1M_TOKENS.values()):
    cost_df = pd.DataFrame([
        {"model": m, "input_per_1m": r["input"], "output_per_1m": r["output"]}
        for m, r in COST_PER_1M_TOKENS.items()
    ]).set_index("model").reindex(MODELS)
    # Blended cost assuming roughly 3 input tokens per output token for a judging call
    # (the judge reads a long prompt and rubric, then writes a short score and reason).
    cost_df["blended_per_1m_3to1"] = (
        3 * cost_df["input_per_1m"] + cost_df["output_per_1m"]
    ) / 4
    display(cost_df.round(4))
else:
    print("Set COST_PER_1M_TOKENS above to compare cost across models.")


### Visual comparison

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
mae["overall"].plot.bar(ax=axes[0], color="steelblue")
axes[0].set_title("Overall MAE (lower = better)")
axes[0].set_ylabel("mean absolute error")
lat_summary["mean_latency_s"].plot.bar(ax=axes[1], color="indianred")
axes[1].set_title("Mean latency per call")
axes[1].set_ylabel("seconds")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## Recommendation
 In the sample runs here, `gpt-4.1-nano` came out ahead on accuracy, latency, and cost, so it is worth considering as a replacement for `gpt-4o-mini` in [`provider.py`](../../../../../src/providers/openai/trulens/providers/openai/provider.py).

